In [5]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re
import json


In [2]:
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

train_data = dataset["train"]

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [3]:
data = []

for item in train_data:
    instruction = item.get("instruction", "").strip()
    response = item.get("response", "").strip()

    # Ensure both fields exist
    if instruction and response:

        text = f"User: {instruction}\nBot: {response}"

        data.append({
            "text": text,
            "source": "bitext",
            "category": item.get("category", "").strip(),
            "intent": item.get("intent", "").strip()
        })

print("Total formatted records:", len(data))
print("Sample record:")
print(data[0])

Total formatted records: 26872
Sample record:
{'text': "User: question about cancelling order {{Order Number}}\nBot: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.", 'source': 'bitext', 'category': 'ORDER', 'intent': 'cancel_order'}


In [4]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-m3")

c:\Users\SEIF\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

c:\Users\SEIF\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SEIF\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [6]:
def normalize(text):
    text = re.sub(r'\{\{[^}]+\}\}', '[ENTITY]', str(text))
    text = re.sub(r'(\[ENTITY\]\s*)+', '[ENTITY] ', text)
    return text.strip()

df = pd.DataFrame(dataset["train"])
df["id"] = [f"rec_{i:05d}" for i in range(len(df))]
df["composite_text"] = df.apply(
    lambda row: f"{row['intent']} {row['category']} {normalize(row['instruction'])}",
    axis=1
)


In [7]:
embed_model = SentenceTransformer("BAAI/bge-m3")
vectors_np = embed_model.encode(
    df["composite_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

dim = vectors_np.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(vectors_np)

c:\Users\SEIF\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/420 [00:00<?, ?it/s]

In [8]:
id_map = {str(i): rec_id for i, rec_id in enumerate(df["id"])}

faiss.write_index(index, "vector_index.faiss")

with open("id_map.json", "w") as f:
    json.dump(id_map, f)

df[["id", "flags", "category", "intent", "composite_text", "response"]].to_csv(
    "records.csv", index=False
)

In [9]:
records = pd.read_csv("records.csv").set_index("id")

In [10]:
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\SEIF\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SEIF\.cache\huggingface\hub\models--google--flan-t5-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
chat_history = []
def ask_rag(question, score_threshold=0.45):

    # 1. Normalize + embed
    normalized_question = normalize(question)
    q_vector = embed_model.encode(
        [normalized_question],
        normalize_embeddings=True
    ).astype("float32").reshape(1, -1)

    # 2. FAISS search — top 1 only
    distances, indices = index.search(q_vector, 1)

    top_score = float(distances[0][0])
    top_idx = int(indices[0][0])

    print(f"[DEBUG] Score: {top_score:.4f}")  # temporary, remove later

    # 3. Confidence check
    if top_score < score_threshold:
        answer = "I don't have enough information to answer that."
        chat_history.append({"question": question, "answer": answer})
        return answer

    # 4. Fetch response directly — no LLM
    rec_id = id_map[str(top_idx)]
    record = records.loc[rec_id]

    # Clean ALL placeholder variants
    response = record["response"]
    response = re.sub(r'\$?\{\{[^}]+\}\}', '[please provide details]', response)
    response = re.sub(r'\[ENTITY\]', '[please provide details]', response)

    chat_history.append({"question": question, "answer": response})
    return response




In [12]:
def debug_retrieval(question, k=5):
    normalized_question = normalize(question)
    q_vector = embed_model.encode(
        [normalized_question],
        normalize_embeddings=True
    ).astype("float32").reshape(1, -1)

    distances, indices = index.search(q_vector, k)

    print(f"Query: '{question}'\n")
    for rank, (dist, i) in enumerate(zip(distances[0], indices[0])):
        rec_id = id_map[str(i)]
        record = records.loc[rec_id]
        print(f"  Rank {rank+1} | Score: {dist:.4f}")
        print(f"  Composite : {record['composite_text']}")
        print(f"  Intent    : {record['intent']}")
        print(f"  Response  : {record['response'][:80]}...")
        print()

In [13]:
debug_retrieval("where is my order")

Query: 'where is my order'

  Rank 1 | Score: 0.7850
  Composite : place_order ORDER where can i order somedthing
  Intent    : place_order
  Response  : We appreciate your inquiry and showing interest in our products! To place an ord...

  Rank 2 | Score: 0.7803
  Composite : place_order ORDER where do I order something?
  Intent    : place_order
  Response  : I'm delighted to assist you in finding the right place to order our products! To...

  Rank 3 | Score: 0.7591
  Composite : place_order ORDER where coujld I order several of your product?
  Intent    : place_order
  Response  : Thank you for your interest in ordering several of our products! We appreciate y...

  Rank 4 | Score: 0.7536
  Composite : place_order ORDER where can i order a few of ur product
  Intent    : place_order
  Response  : Unquestionably! I'm thrilled to see your interest in ordering a few of our produ...

  Rank 5 | Score: 0.7521
  Composite : place_order ORDER where can i shop something
  Intent    : place

In [14]:
debug_retrieval("I want to refund my payment")

Query: 'I want to refund my payment'

  Rank 1 | Score: 0.8484
  Composite : get_refund REFUND wanna ask for a restutution of my money
  Intent    : get_refund
  Response  : I get that you're seeking a restitution of your money, and I'm here to assist yo...

  Rank 2 | Score: 0.8484
  Composite : get_refund REFUND I'm trying to request my money back
  Intent    : get_refund
  Response  : I certainly recognize your desire to request a refund for your money. It can be ...

  Rank 3 | Score: 0.8364
  Composite : get_refund REFUND i need help to ask for a refund of my money
  Intent    : get_refund
  Response  : I've picked up that you're seeking help with requesting a refund for your money....

  Rank 4 | Score: 0.8347
  Composite : get_refund REFUND need assistance demanding my money back
  Intent    : get_refund
  Response  : I've got that you require assistance in demanding a refund of your money. I'm he...

  Rank 5 | Score: 0.8287
  Composite : get_refund REFUND I need to demand my m

In [15]:
print(ask_rag("what is my order status"))

[DEBUG] Score: 0.8079
Thank you for contacting us! I'm here to assist you in checking the current status of purchase [please provide details]. To view the status of your purchase, please visit the '[please provide details]' section on our website. If you have any other questions or need further assistance, feel free to ask. We're dedicated to providing you with a seamless experience!


In [16]:
print(chat_history)

[{'question': 'what is my order status', 'answer': "Thank you for contacting us! I'm here to assist you in checking the current status of purchase [please provide details]. To view the status of your purchase, please visit the '[please provide details]' section on our website. If you have any other questions or need further assistance, feel free to ask. We're dedicated to providing you with a seamless experience!"}]


In [17]:
# See all unique intents
print(df["intent"].value_counts().to_string())

intent
check_invoice               1000
complaint                   1000
contact_customer_service    1000
edit_account                1000
switch_account              1000
check_payment_methods        999
contact_human_agent          999
delivery_period              999
get_invoice                  999
newsletter_subscription      999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
set_up_shipping_address      997
delete_account               995
delivery_options             995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950


In [18]:
# Save the T5 model and tokenizer to a local directory
model.save_pretrained("./saved_t5_model")
tokenizer.save_pretrained("./saved_t5_model")

('./saved_t5_model\\tokenizer_config.json',
 './saved_t5_model\\special_tokens_map.json',
 './saved_t5_model\\tokenizer.json')